# Etapa 2 (Análise)

In [1]:
# criando seção spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("INPE-MLlib")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

d:\data_science\inpe-mllib-study\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Após executar o notebook `1_data_preparation.ipynb`, vamos ler os dados que foram salvos para começar a parte da análise do modelo

In [8]:
df = spark.read.parquet(
    "../data/bronze/inpe_inmet/"
)

Documentando o dataset:

In [9]:
df.printSchema()

root
 |-- id_foco: long (nullable = true)
 |-- data_hora_foco: timestamp (nullable = true)
 |-- satelite: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- bioma: string (nullable = true)
 |-- frp: double (nullable = true)
 |-- latitude_foco: double (nullable = true)
 |-- longitude_foco: double (nullable = true)
 |-- codigo_estacao: string (nullable = true)
 |-- nome_estacao: string (nullable = true)
 |-- latitude_estacao: double (nullable = true)
 |-- longitude_estacao: double (nullable = true)
 |-- distancia_estacao_km: double (nullable = true)
 |-- temperatura_ar_c: double (nullable = true)
 |-- umidade_relativa_ar_pct: double (nullable = true)
 |-- precipitacao_total_horario_mm: double (nullable = true)
 |-- vento_velocidade_horaria_ms: double (nullable = true)
 |-- radiacao_global_kj_m2: double (nullable = true)
 |-- ano: integer (nullable = true)



Temos 20 colunas e os tipos estão corretos porque tratamos antes na parte de preparação dos dados

In [10]:
df.count()

8951835

Temos 8951835 registros

In [11]:
df.describe().show()

+-------+--------------------+---------+---------+-------------+--------------+-----------------+------------------+------------------+--------------+------------+-------------------+------------------+--------------------+------------------+-----------------------+-----------------------------+---------------------------+---------------------+------------------+
|summary|             id_foco| satelite|   estado|    municipio|         bioma|              frp|     latitude_foco|    longitude_foco|codigo_estacao|nome_estacao|   latitude_estacao| longitude_estacao|distancia_estacao_km|  temperatura_ar_c|umidade_relativa_ar_pct|precipitacao_total_horario_mm|vento_velocidade_horaria_ms|radiacao_global_kj_m2|               ano|
+-------+--------------------+---------+---------+-------------+--------------+-----------------+------------------+------------------+--------------+------------+-------------------+------------------+--------------------+------------------+-----------------------+--

In [24]:
df.show()

+------------+-------------------+--------+----------+--------------------+--------------+-----+------------------+------------------+--------------+----------------+----------------+-----------------+--------------------+----------------+-----------------------+-----------------------------+---------------------------+---------------------+----+------+
|     id_foco|     data_hora_foco|satelite|    estado|           municipio|         bioma|  frp|     latitude_foco|    longitude_foco|codigo_estacao|    nome_estacao|latitude_estacao|longitude_estacao|distancia_estacao_km|temperatura_ar_c|umidade_relativa_ar_pct|precipitacao_total_horario_mm|vento_velocidade_horaria_ms|radiacao_global_kj_m2| ano|target|
+------------+-------------------+--------+----------+--------------------+--------------+-----+------------------+------------------+--------------+----------------+----------------+-----------------+--------------------+----------------+-----------------------+-------------------------

Interessante é perceber que em relação a variável alvo, a média é 33, enquanto o valor máximo é 9612, provavelmente temos poucos casos com esse valor máximo. Outra coisa interessante é que a precipitação média é quase 0, ou seja, tem muito mais horas sem chover do que chovendo, o que é esperado. O valor da velocidade do vento tem média 1 com dp de 1.5, enquanto o valor máximo é 13, mostrando mais uma vez essa discrepância, já a radiação global parece mais correta.

Analisando a distribuição da variável alvo (FRP):

In [ ]:
# vamos criar a variável alvo (1 se maior que a mediana de frp e vice-versa)
# primeiro pegando a mediana (quantil 0.5)
from pyspark.sql import functions as F

mediana_frp = df.approxQuantile("FRP", [0.5], 0.001)[0]

print(mediana_frp)

9.0


In [16]:
# agora criando a variável binária

df = df.withColumn(
    'target',
    F.when(F.col('FRP') > mediana_frp, 1).otherwise(0)
)

df.select('target').show(5)

+------+
|target|
+------+
|     0|
|     0|
|     0|
|     1|
|     1|
+------+
only showing top 5 rows


Verificando a proporção de cada classe:

In [22]:
total = df.count()

distribuicao = (
    df.groupBy('target')
    .count()
    .withColumn(
        "proporcao",
        F.round(F.col("count")/F.lit(total), 3)
    )
)

distribuicao.show()

+------+-------+---------+
|target|  count|proporcao|
+------+-------+---------+
|     1|3655499|    0.408|
|     0|5296336|    0.592|
+------+-------+---------+



Percebemos que temos muito mais valores abaixo da mediana (target = 0) com aproximadamente 59,2% enquanto temos menos valores acima da mediana (target = 1), com 40,8%, o que é esperado considerando que temos mais focos de incendio menores do que maiores.

Analisando o percentual de valores nulos por coluna:

In [23]:
total = df.count()

df.select([
    F.round(
        F.sum(F.col(c).isNull().cast("int")) / F.lit(total) * 100,
        2
    ).alias(c)
    for c in df.columns
]).show()

+-------+--------------+--------+------+---------+-----+-----+-------------+--------------+--------------+------------+----------------+-----------------+--------------------+----------------+-----------------------+-----------------------------+---------------------------+---------------------+---+------+
|id_foco|data_hora_foco|satelite|estado|municipio|bioma|  frp|latitude_foco|longitude_foco|codigo_estacao|nome_estacao|latitude_estacao|longitude_estacao|distancia_estacao_km|temperatura_ar_c|umidade_relativa_ar_pct|precipitacao_total_horario_mm|vento_velocidade_horaria_ms|radiacao_global_kj_m2|ano|target|
+-------+--------------+--------+------+---------+-----+-----+-------------+--------------+--------------+------------+----------------+-----------------+--------------------+----------------+-----------------------+-----------------------------+---------------------------+---------------------+---+------+
|    0.0|           0.0|     0.0|   0.0|      0.0|  0.0|18.11|          0.0|

Primeiramente, olhando para a nossa variável alvo, o FRP, temos aproximadamente 18% dos dados faltantes. Isso é um ponto importante, principalmente porque usamos o FRP para criar a nossa target binária, onde valores acima da mediana recebem 1 e valores abaixo ou iguais recebem 0. O problema é que valores nulos podem acabar sendo classificados como 0 mesmo sem sabermos o valor real do FRP. Por isso, precisamos tratar esses casos antes de calcular a mediana e criar a target. Como o FRP é justamente a variável que queremos prever, acredito que o mais adequado seja remover esses registros, ao invés de tentar preencher os valores faltantes com média ou mediana.

Olhando agora para precipitação e umidade, que são muito importantes para a nossa hipótese principal, temos aproximadamente 31,5% de dados faltantes para precipitação e 27,5% para umidade. Apesar de serem valores consideráveis, ainda temos a maior parte dos dados disponíveis, então podemos avaliar alguma estratégia de preenchimento, como o uso da mediana. Antes disso, porém, seria interessante verificar se esses valores faltantes estão concentrados em determinadas estações ou períodos. No caso da precipitação, precisamos ter um cuidado maior, pois um valor nulo não significa necessariamente que não houve chuva.

Para a primeira hipótese secundária, o problema acaba sendo parecido, já que ela também depende da precipitação e da umidade para identificar regiões mais secas e verificar se existe uma maior concentração de focos com target = 1 nessas áreas. Portanto, a forma como esses valores faltantes forem tratados pode afetar diretamente essa análise, sendo importante definir esse tratamento antes de realizar a parte espacial.

Já para a segunda hipótese secundária temos uma limitação bem maior, pois aproximadamente 86,1% dos valores de velocidade do vento estão faltando. Nesse caso, preencher os dados usando média ou mediana provavelmente não seria uma boa opção, já que estaríamos estimando artificialmente a maior parte da coluna. Uma possibilidade é verificar se esses valores faltantes estão concentrados em determinadas estações ou períodos. Caso existam estações com uma quantidade maior de dados disponíveis, podemos considerar realizar essa análise apenas com essa parte dos dados.

Por fim, temperatura do ar e radiação global possuem aproximadamente 92,7% e 93,7% de valores faltantes, respectivamente. Como essas variáveis não são centrais para as nossas hipóteses e possuem poucos dados disponíveis, acredito que faça mais sentido considerar a retirada delas das análises e dos modelos, ao invés de tentar preencher mais de 90% de seus valores.


Falando dos dados em si, temos uma granularidade de um foco com uma estação mais próxima associada, tendo informações de FRP (Fire Radiative Power), precipitação, umidade e etc, e assim podemos trabalhar com isso para testar as hipóteses e construir o modelo de classificação

# Etapa 3

Tarefa: Filtrar registros inválidos ou irrelevantes, documentando cada filtro e quantos registros foram removidos

A primeira coisa que chama atenção aqui é o FRP negativo, isso é inválido e pode ser considerado um filtro, vamos verificar quantos registros somem:

In [25]:
df.count()

8951835

In [ ]:
# aplicando o filtro

df.filter(F.col("frp") <= 0).count()

848

In [27]:
df.filter(F.col("frp") <= 0) \
  .groupBy("frp") \
  .count() \
  .orderBy("frp") \
  .show(50)

+----+-----+
| frp|count|
+----+-----+
|-1.3|    1|
|-1.1|    2|
|-1.0|    1|
|-0.9|    1|
| 0.0|  843|
+----+-----+



In [28]:
df.groupBy("estado").count().orderBy("estado").show()

+-------------------+-------+
|             estado|  count|
+-------------------+-------+
|            ALAGOAS|  44542|
|              BAHIA|1264509|
|              CEARÁ| 383202|
|   DISTRITO FEDERAL|   2033|
|     ESPÍRITO SANTO|  50194|
|              GOIÁS| 243196|
|           MARANHÃO|2366934|
|        MATO GROSSO|  13754|
|       MINAS GERAIS| 335598|
|            PARAÍBA|  97999|
|               PARÁ|1303470|
|         PERNAMBUCO| 141339|
|              PIAUÍ|1351724|
|RIO GRANDE DO NORTE|  63480|
|            SERGIPE|  20975|
|          TOCANTINS|1268886|
+-------------------+-------+

